In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # lets a repository clone find flybrain without installing it
from flybrain import FlyBrain

brain = FlyBrain(device="auto")  # uses an NVIDIA GPU if CuPy is installed, else the CPU
print(f"{brain.n:,} neurons, {len(brain.indices):,} connections, running on {brain.device}")

In [ ]:
classes, counts = np.unique(brain.superclass, return_counts=True)
for c, k in sorted(zip(classes, counts), key=lambda x: -x[1])[:12]:
    print(f"{k:7,}  {c}")

In [ ]:
loom_left = brain.cells(["LC4", "LPLC2"], side="L")
giant_fiber = {s: brain.cells(["DNp01"], side=s) for s in "LR"}
print(f"stimulating {len(loom_left)} looming-detector neurons on the left")

def record(stim_cells, strength=0.8, seconds=3.0, on=(1.0, 2.0), seed=0, watch=giant_fiber):
    """Run the brain, stimulating `stim_cells` during `on`. Returns spike times per watched group
    and each neuron's spike count while the stimulus was on."""
    brain.reset(seed)
    spikes = {k: [] for k in watch}
    counts = np.zeros(brain.n, np.int32)
    for step in range(int(seconds / brain.dt)):
        t = step * brain.dt
        if on[0] <= t < on[1]:
            brain.stimulate(stim_cells, strength)
        fired = brain.step()
        if on[0] <= t < on[1]:
            counts[fired] += 1
        for k, idx in watch.items():
            spikes[k] += [t] * int(np.isin(idx, fired).sum())
    return spikes, counts

spikes, counts = record(loom_left)
fig, ax = plt.subplots(figsize=(8, 2.2))
for row, (side, times) in enumerate(spikes.items()):
    ax.eventplot([times], lineoffsets=row, colors="#e0342c" if side == "L" else "#2a9d8f")
ax.axvspan(1, 2, color="0.9", zorder=0)
ax.set_yticks([0, 1], ["DNp01 left", "DNp01 right"])
ax.set_xlabel("time (s); grey = left looming detectors stimulated")
ax.set_title("The left giant fiber fires; the right one stays quiet")
plt.show()

In [ ]:
_, rest_counts = record(np.array([], int))
active = counts - rest_counts >= 2
ok = ~np.isnan(brain.positions).any(axis=1)
xy = brain.positions[:, [0, 2]]
if np.nanmean(xy[brain.side == "R", 0]) < np.nanmean(xy[brain.side == "L", 0]):
    xy = xy * [-1, 1]  # draw the fly's right on the viewer's right
fig, ax = plt.subplots(figsize=(5, 8))
ax.scatter(*xy[ok & ~active].T, s=0.05, c="0.8", rasterized=True)
ax.scatter(*xy[ok & active].T, s=1.5, c="#e0342c", rasterized=True)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"{active.sum():,} neurons driven by the left looming detectors")
plt.show()

In [ ]:
chase_left = brain.cells(["LC10a"], side="L")
steer = {s: brain.cells(["DNa02"], side=s) for s in "LR"}

def rates(stim, seeds=range(4)):
    out = {s: [] for s in "LR"}
    for seed in seeds:
        spikes, _ = record(stim, seconds=2.5, on=(0.5, 2.5), seed=seed, watch=steer)
        for s in "LR":
            out[s].append(len([t for t in spikes[s] if t >= 0.5]) / 2.0)
    return {s: float(np.mean(v)) for s, v in out.items()}

rest, chase = rates(np.array([], int)), rates(chase_left)
fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(2)
ax.bar(x - 0.2, [rest["L"], rest["R"]], 0.4, label="no stimulus", color="0.7")
ax.bar(x + 0.2, [chase["L"], chase["R"]], 0.4, label="left LC10a stimulated", color="#e0342c")
ax.set_xticks(x, ["DNa02 left", "DNa02 right"])
ax.set_ylabel("spikes / s")
ax.legend(frameon=False)
plt.show()
print("rest :", {k: round(v, 1) for k, v in rest.items()})
print("chase:", {k: round(v, 1) for k, v in chase.items()})


In [ ]:
import numpy as np

# Find the left and right LC10a neurons
left_lc10a = brain.cells(["LC10a"], side="L")
right_lc10a = brain.cells(["LC10a"], side="R")

# Find the left and right DNa02 output neurons
left_dna02 = brain.cells(["DNa02"], side="L")
right_dna02 = brain.cells(["DNa02"], side="R")

print("Left LC10a:", len(left_lc10a))
print("Right LC10a:", len(right_lc10a))
print("Left DNa02:", len(left_dna02))
print("Right DNa02:", len(right_dna02))

In [ ]:
def run_experiment(side):
    brain.reset(seed=42)

    if side == "L":
        stimulus = left_lc10a
    else:
        stimulus = right_lc10a

    # Apply the stimulus
    brain.stimulate(stimulus, 0.8)

    left_spikes = 0
    right_spikes = 0
    steps = 100  # 100 steps x 20 ms = 2 seconds

    for _ in range(steps):
        fired = brain.step()
        left_spikes += np.isin(fired, left_dna02).sum()
        right_spikes += np.isin(fired, right_dna02).sum()

    # Convert counts to approximate firing rates
    duration = steps * 0.02

    return {
        "left_DNa02": left_spikes / duration,
        "right_DNa02": right_spikes / duration,
    }

print("Left stimulus:", run_experiment("L"))
print("Right stimulus:", run_experiment("R"))

In [ ]:
chase_right = brain.cells(["LC10a"], side="R")

rest_rates = rates(np.array([], int))
left_rates = rates(chase_left)
right_rates = rates(chase_right)

print("Rest:", rest_rates)
print("Left LC10a stimulus:", left_rates)
print("Right LC10a stimulus:", right_rates)

In [ ]:
# Repeat the same experiment across 20 seeds
seeds = range(20)

rest_20 = rates(np.array([], int), seeds=seeds)
left_20 = rates(chase_left, seeds=seeds)
right_20 = rates(chase_right, seeds=seeds)

print("Rest:", rest_20)
print("Left LC10a stimulus:", left_20)
print("Right LC10a stimulus:", right_20)

In [ ]:
# Test how stimulation strength affects DNa02 firing
stimulus_levels = [0.2, 0.4, 0.6, 0.8, 1.0]
steer = {s: brain.cells(["DNa02"], side=s) for s in "LR"}

def rates_at_strength(stim_cells, strength, seeds=range(20)):
    out = {s: [] for s in "LR"}

    for seed in seeds:
        spikes, _ = record(
            stim_cells,
            strength=strength,
            seconds=2.5,
            on=(0.5, 2.5),
            seed=seed,
            watch=steer
        )

        for side in "LR":
            out[side].append(
                sum(t >= 0.5 for t in spikes[side]) / 2.0
            )

    return {
        side: float(np.mean(values))
        for side, values in out.items()
    }

for amount in stimulus_levels:
    left_result = rates_at_strength(chase_left, amount)
    right_result = rates_at_strength(chase_right, amount)

    print(f"\nStrength: {amount}")
    print("  Left LC10a:", left_result)
    print("  Right LC10a:", right_result)

In [ ]:
print("Input-related methods:")
print([name for name in dir(brain)
       if any(word in name.lower()
              for word in ["stim", "input", "visual", "cell", "step", "reset"])])

In [ ]:
sensory_input:
  Type: <class 'bool'>
  Callable: False
  Value: True

visual:
  Type: <class 'numpy.ndarray'>
  Callable: False
  Value: array([  1053,   5045,   5173, ..., 166120, 166159, 166178],
      shape=(6006,), dtype=int32)

_visual:
  Type: <class 'numpy.ndarray'>
  Callable: False
  Value: array([  1053,   5045,   5173, ..., 166120, 166159, 166178],
      shape=(6006,), dtype=int32)

stimulate:
  Type: <class 'method'>
  Callable: True
  Signature: (idx: 'np.ndarray', amount) -> 'None'
  Documentation: Add voltage to these neurons right now (before the next step).

step:
  Type: <class 'method'>
  Callable: True
  Signature: (eye_drive: 'np.ndarray | None' = None, inject=())
  Documentation: Advance one step (dt, 20 ms by default). eye_drive: 0..1 per photoreceptor (len(self.visual)), or
(len(self.visual), batch); inject: (neuron indices, extra voltage) pairs
added this step. Returns the indices of the neurons that fired (NumPy):
one array with batch 1, else a list with one array per fly.

In [ ]:
for name in ["sensory_input", "visual", "_visual", "stimulate", "step"]:
    obj = getattr(brain, name)
    print(f"\n{name}:")
    print("  Type:", type(obj))
    print("  Callable:", callable(obj))

    if callable(obj):
        print("  Signature:", inspect.signature(obj))
        print("  Documentation:", inspect.getdoc(obj))
    else:
        print("  Value:", repr(obj)[:300])

In [ ]:
import numpy as np

def test_visual_input(level, seed=42):
    brain.reset(seed=seed)

    # One input value for each of the 6,006 photoreceptors
    eye_drive = np.full(
        len(brain.visual), level, dtype=np.float32
    )

    total_fired = 0

    # Run for 1 second (50 steps × 20 ms)
    for _ in range(50):
        fired = brain.step(eye_drive=eye_drive)
        total_fired += len(fired)

    return total_fired

print("Zero visual input:", test_visual_input(0.0))
print("Half-strength input:", test_visual_input(0.5))
print("Full-strength input:", test_visual_input(1.0))

In [ ]:
import inspect

print("Visual-related attributes:")
for name in dir(brain):
    if "visual" in name.lower() or "eye" in name.lower():
        obj = getattr(brain, name)
        print(f"\n{name}: {type(obj)}")
        if callable(obj):
            try:
                print(inspect.signature(obj))
                print(inspect.getdoc(obj))
            except (TypeError, ValueError):
                pass

In [ ]:
print("Number of visual neurons:", len(brain.visual))
print("First 30 visual indices:")
print(brain.visual[:30])

print("\nEye gain:", brain.eye_gain)

print("\nBrain attributes related to geometry or visual layout:")
print([
    name for name in dir(brain)
    if any(word in name.lower()
           for word in ["pos", "coord", "optic", "retina", "eye", "column", "row"])
])

In [ ]:
print("Positions type:", type(brain.positions))
print("Positions shape:", brain.positions.shape)
print("First 10 positions:")
print(brain.positions[:10])

print("\nVisual neuron indices:", brain.visual[:10])
print("Positions of first 10 visual neurons:")
print(brain.positions[brain.visual[:10]])

In [ ]:
import inspect

print(inspect.getsource(type(brain).step))

In [ ]:
import numpy as np

n_visual = len(brain.visual)

# Three input patterns
no_input = np.zeros(n_visual, dtype=np.float32)

uniform_input = np.full(
    n_visual, 0.5, dtype=np.float32
)

partial_input = np.zeros(n_visual, dtype=np.float32)
partial_input[:n_visual // 4] = 0.5


def measure_activity(eye_drive, seed=42):
    brain.reset(seed=seed)
    total_events = 0

    for _ in range(50):  # 1 second at 20 ms per step
        fired = brain.step(eye_drive=eye_drive)
        total_events += len(fired)

    return total_events


print("No input:", measure_activity(no_input))
print("Uniform input:", measure_activity(uniform_input))
print("Partial input:", measure_activity(partial_input))

In [ ]:
import numpy as np

def measure_visual_response(eye_drive, seed=42):
    brain.reset(seed=seed)
    visual_events = 0
    other_events = 0

    for _ in range(50):
        fired = brain.step(eye_drive=eye_drive)

        visual_events += np.isin(fired, brain.visual).sum()
        other_events += len(fired) - np.isin(fired, brain.visual).sum()

    return int(visual_events), int(other_events)


n = len(brain.visual)

no_input = np.zeros(n, dtype=np.float32)
uniform_input = np.full(n, 0.5, dtype=np.float32)
partial_input = np.zeros(n, dtype=np.float32)
partial_input[:n // 4] = 0.5

for name, pattern in [
    ("No input", no_input),
    ("Uniform", uniform_input),
    ("Partial", partial_input),
]:
    visual, other = measure_visual_response(pattern)
    print(f"{name}: visual events={visual}, other events={other}")

In [ ]:
import numpy as np

steer = {
    side: brain.cells(["DNa02"], side=side)
    for side in "LR"
}

def measure_steering(eye_drive, seed=42):
    brain.reset(seed=seed)
    counts = {"L": 0, "R": 0}

    for _ in range(50):
        fired = brain.step(eye_drive=eye_drive)

        for side in "LR":
            counts[side] += np.isin(
                fired, steer[side]
            ).sum()

    return counts

n = len(brain.visual)

patterns = {
    "No input": np.zeros(n, dtype=np.float32),
    "Uniform": np.full(n, 0.5, dtype=np.float32),
    "Partial": np.concatenate([
        np.full(n // 4, 0.5, dtype=np.float32),
        np.zeros(n - n // 4, dtype=np.float32)
    ])
}

for name, pattern in patterns.items():
    print(name, measure_steering(pattern))

In [ ]:
chase = {
    side: brain.cells(["LC10a"], side=side)
    for side in "LR"
}

def measure_lc10a(eye_drive, seed=42):
    brain.reset(seed=seed)
    counts = {"L": 0, "R": 0}

    for _ in range(50):
        fired = brain.step(eye_drive=eye_drive)

        for side in "LR":
            counts[side] += np.isin(
                fired, chase[side]
            ).sum()

    return counts

n = len(brain.visual)

patterns = {
    "No input": np.zeros(n, dtype=np.float32),
    "Uniform": np.full(n, 0.5, dtype=np.float32),
    "Partial": np.concatenate([
        np.full(n // 4, 0.5, dtype=np.float32),
        np.zeros(n - n // 4, dtype=np.float32)
    ])
}

for name, pattern in patterns.items():
    print(name, measure_lc10a(pattern))